<h1 style="color: #4166c4; font-weight: 700;">Preprocessing</h1>

## Table of Contents

- [1. Introduction](#introduction)
- [2. Metadata preprocessing](#metadata-preprocessing)
  - [2.1 Correcting inconsistencies](#correcting-inconsistencies)
    - [2.1.1 Correcting artist names](#artist-names)
      - [2.1.1.1 Infer artist names from title structure](#title-inference)
      - [2.1.1.2 Review artist labels containing `Genius`](#genius-review)
    - [2.1.2 Correcting years](#release-years)
  - [2.2 Lyrics normalization](#lyrics-normalization)
  - [2.3 Duplicate-lyrics audit](#duplicate-audit)

<h2 style="color: #4166c4; font-weight: 400;">1. Introduction <a id="introduction"></a></h2>


This notebook independently prepares the supplied training and test datasets by reviewing and correcting likely artist-name and year inconsistencies, then normalizing lyric text. Run it from the project root so the `data/` paths resolve. The cells load `data/train.csv`, `data/test.csv`, and `data/sample_submission.csv`; the corrected frames remain available as `X_train` and `X_test` during this notebook session.

Artist and year corrections use Genius lookups for unresolved candidates. Those lookups may take time or fail when data is unavailable; the code reports unresolved entries rather than inventing values. Keep the original CSV files unchanged and save corrected outputs explicitly if you need persistent copies.

In [1]:
import pandas as pd
df_train = pd.read_csv('data/train.csv')

In [ ]:
from preprocessing import fix_all_translation_artists, correct_pre_years
from utils import get_info_batch


In [3]:
X_train = df_train.drop(columns=['tag'])
y_train = df_train['tag']

In [4]:
X_test=pd.read_csv('data/test.csv')
y_test=pd.read_csv('data/sample_submission.csv')

<h2 style="color: #4166c4; font-weight: 400;">2. Metadata preprocessing <a id="metadata-preprocessing"></a></h2>

<h3 style="color: #597edb; font-weight: 400;">2.1 Correcting inconsistencies <a id="correcting-inconsistencies"></a></h3>

<h4>2.1.1 Correcting artist names <a id="artist-names"></a></h4>

Some records list a Genius translation, transliteration, or romanization account as the artist. The following steps identify likely platform-account labels in the training and test frames, first infer an artist from a title when its format allows, and then query Genius for remaining candidates. Unresolved names are left unchanged for review; a match on the word `Genius` alone is not sufficient evidence that an artist label is wrong.

Identify likely Genius translation or romanization account names and inspect their titles. These are candidates for correction, not confirmed errors; review the displayed examples before relying on inferred names.

In [5]:
translation_account_pattern = (
    r'(?i)\bgenius\b.*\b(?:translat\w*|trad\w*|romaniz\w*|romanis\w*|transliter\w*)\b')

In [6]:
train_translation_mask = X_train['artist'].str.contains(
    translation_account_pattern, na=False, regex=True
) #checking which rows have artists that are translation accounts in train
test_translation_mask = X_test['artist'].str.contains(
    translation_account_pattern, na=False, regex=True
) #checking which rows have artists that are translation accounts in test

#extracting this specififc info about those rows so we can visualize them and chek if there's a pattern
train_miss_artists = X_train.loc[
    train_translation_mask, ['id', 'title', 'artist']
] 
test_miss_artists = X_test.loc[
    test_translation_mask, ['id', 'title', 'artist']
]


print(f"Train: {len(train_miss_artists)} rows")
display(train_miss_artists['artist'].value_counts().rename_axis('artist').reset_index(name='count'))
display(train_miss_artists.head(10))

print(f"Test: {len(test_miss_artists)} rows")
display(test_miss_artists['artist'].value_counts().rename_axis('artist').reset_index(name='count'))
display(test_miss_artists.head(10))

Train: 606 rows


,artist,count
0,Genius English Translations,555
1,Genius Traductions Franaises,21
2,Genius Romanizations,13
3,Genius Brasil Tradues,9
4,Genius South Africa Translations,3
5,Genius Traduzioni Italiane,2
6,Genius Farsi Translations ( ),1
7,Genius Swedish Translations,1
8,Genius Translations,1


,id,title,artist
116,6815917,Ferbinho - Lionel English Translation,Genius English Translations
449,7634567,Def. - WANT U English Translation,Genius English Translations
459,7550031,Sogumm 소금 Dress - Honey bee ft. PENOMECO 페노메코 ...,Genius English Translations
526,7749379,Ricky Martin - Náufrago English Translation,Genius English Translations
617,3506177,Drake - Legend Tradução em Português,Genius Brasil Tradues
926,7366990,Coogie - Swipe Feat. Ourealgoat Touch the Sky...,Genius English Translations
947,4477095,Chanyeol EXO - 봄 여름 가을 겨울 SSFW English Transla...,Genius English Translations
1373,6706099,Orkundk - Marsa Gönder Beni Elon Musk English ...,Genius English Translations
1835,5679491,Eraserheads - Minsan English Translation,Genius English Translations
2228,6564362,Damons Year - August,Genius English Translations


Test: 122 rows


,artist,count
0,Genius English Translations,117
1,Genius Brasil Tradues,2
2,Genius Swedish Translations,1
3,Genius Romanizations,1
4,Genius Traductions Franaises,1


,id,title,artist
406,7349077,Hov1 - När jag ser dig English Translation,Genius Swedish Translations
814,4691789,Гречка Grechka – Люби меня люби Love Me Love M...,Genius English Translations
847,5449591,Nu Drama - Ikigai English Translation,Genius English Translations
930,7667325,中川 翔子 Shoko Nakagawa - 空色デイズ Sorairo Days Engl...,Genius English Translations
1207,7519904,10CM - 서랍 Drawer English Translation,Genius English Translations
2635,5573039,Mahmood - Moonlight popolare ft. Massimo Peric...,Genius English Translations
3013,7252584,APRIL - Oh my mistake English Translation,Genius English Translations
3544,5614170,Anuel AA - Nubes Negras English Translation,Genius English Translations
3898,7602975,Moon Byul - 너만 들었으면 좋겠다 For Me English Transla...,Genius English Translations
4661,498735,Drake - Karaoke Tradução em Português,Genius Brasil Tradues


<h5>Infer artist names from title structure <a id="title-inference"></a></h5>

In [7]:

X_train = fix_all_translation_artists(X_train)
X_test = fix_all_translation_artists(X_test)

#checking how many rows couldn't be fixed by the structure approach
train_remaining_translation_mask = X_train['artist'].str.contains(
    translation_account_pattern, na=False, regex=True
)
test_remaining_translation_mask = X_test['artist'].str.contains(
    translation_account_pattern, na=False, regex=True
)

print(f"Train remaining translation-account rows: {train_remaining_translation_mask.sum()}")
display(X_train.loc[train_remaining_translation_mask, 'artist'].value_counts().rename_axis('artist').reset_index(name='count'))

print(f"Test remaining translation-account rows: {test_remaining_translation_mask.sum()}")
display(X_test.loc[test_remaining_translation_mask, 'artist'].value_counts().rename_axis('artist').reset_index(name='count'))


Train remaining translation-account rows: 16


,artist,count
0,Genius English Translations,15
1,Genius South Africa Translations,1


Test remaining translation-account rows: 5


,artist,count
0,Genius English Translations,5


In [8]:
#getting those rows that couldnt be corrected by the structure
for name, data in [('Train', X_train), ('Test', X_test)]:
    remaining_mask = data['artist'].str.contains(
        translation_account_pattern, na=False, regex=True
    )
    remaining_ids = data.loc[remaining_mask, 'id']

    print(f"{name}: querying {len(remaining_ids)} remaining translation-account rows...")
    if not remaining_ids.empty:
        fetched = get_info_batch(remaining_ids, artist=True) # trying to correct those artists trough the Genius API
        data.loc[remaining_mask, 'artist'] = (
            data.loc[remaining_mask, 'id'].map(fetched['artist'])
            .where(lambda artists: artists.notna(), data.loc[remaining_mask, 'artist'])
        )
 #checking how many rows couldn't be fixed by the API approach
    still_remaining_mask = data['artist'].str.contains(
        translation_account_pattern, na=False, regex=True
    )
    still_remaining_rows = data.loc[
        still_remaining_mask, ['id', 'title', 'artist']
    ]
    print(f"{name}: remaining translation-account rows after lookup: {len(still_remaining_rows)}")
    display(still_remaining_rows)

Train: querying 16 remaining translation-account rows...
Train: remaining translation-account rows after lookup: 5


,id,title,artist
72559,7388735,Dangote Lyrics English Translation,Genius English Translations
86688,5952033,LLuvia en el Paraiso English Translation,Genius English Translations
111692,3953910,RAINING Korean Ver. English Translation,Genius English Translations
121008,4413755,So What English Translation,Genius English Translations
123064,6963138,Sicilian Panda RMX,Genius English Translations


Test: querying 5 remaining translation-account rows...
Test: remaining translation-account rows after lookup: 1


,id,title,artist
10498,3248493,B.O.C English Translation,Genius English Translations


`fix_all_translation_artists` extracts an artist from titles that follow a recognizable `Artist - Title` pattern. The following lookup attempts to resolve remaining candidates through Genius. Any candidate the title rule or lookup cannot resolve stays unchanged, and the output shows those cases for review.

This check lists artist labels containing `Genius` across both splits. It is a review aid only: legitimate artist names can contain the same word, so do not treat every match as an incorrect platform credit.

<h5> Review artist labels containing `Genius` <a id="genius-review"></a></h5>

In [9]:
# Show all artist values that contain the word "Genius" (case-insensitive)
genius_accounts = (
    pd.concat([X_train['artist'], X_test['artist']])
    .dropna()
    .loc[lambda s: s.str.contains('genius', case=False)]
    .value_counts()
)
print(genius_accounts.to_string())
print(f'total sum: {genius_accounts.sum()}')


artist
Genius                                13
Outside the Lines With Rap Genius     13
Genius Users                          13
Polskie tumaczenia Genius             12
Sports Genius                          6
Genius English Translations            6
Perfume Genius                         5
Rock Genius                            5
Fashion Genius                         5
Rap Genius                             4
Dotard Genius                          4
Game Genius                            3
Screen Genius                          3
Genius Dream Album                     3
Lit Genius                             3
Eugenius (band)                        3
Genius Community                       2
Genius Editors                         2
Genius UK                              2
Genius Lists                           2
DJ Muggs & GZA The Genius              2
Math Genius                            2
News Genius                            2
Genius South Africa                    2
MisUnders

Inspect the values listed above and correct only labels supported by reliable evidence. This notebook leaves ambiguous names unchanged rather than making additional API requests or guessing.

<h4>2.1.2 Correcting years <a id="release-years"></a></h4>

`correct_pre_years` checks records with a year earlier than 1900 and attempts to retrieve a release date from Genius. A returned year of 1800 or later replaces the current value. Earlier API years are classified as suspicious, and records with no usable API year are classified as not found; in both cases the year is set to missing and the song IDs are retained for review. The function returns updated frames, which are assigned back to `X_train` and `X_test`.

In [10]:
X_train, train_years_resolved_ids, train_years_suspicious_ids, train_years_not_found_ids = correct_pre_years(
    1900, X_train, 'Train'
)
X_test, test_years_resolved_ids, test_years_suspicious_ids, test_years_not_found_ids = correct_pre_years(
    1900, X_test, 'Test'
)

Train: 653 rows with years before 1900
Train: accepted 265 years; 160 suspicious; 228 not found
Test: 182 rows with years before 1900
Test: accepted 85 years; 41 suspicious; 56 not found


In [11]:
def summarize_years(frame, ids, split, status):
    years = pd.to_numeric(
        frame.loc[frame['id'].isin(ids), 'year'],
        errors='coerce'
    ).dropna()

    return {
        'split': split,
        'status': status,
        'rows': len(ids),
        'rows_with_year': len(years),
        'minimum_year': years.min() if not years.empty else None,
        'years_below_1900': int((years < 1900).sum()),
        'maximum_year': years.max() if not years.empty else None,
    }




year_summary = pd.DataFrame([
    summarize_years(X_train, train_years_resolved_ids, 'Train', 'Resolved'),
    summarize_years(X_test, test_years_resolved_ids, 'Test', 'Resolved'),
    summarize_years(X_train, train_years_suspicious_ids, 'Train', 'Suspicious'),
    summarize_years(X_test, test_years_suspicious_ids, 'Test', 'Suspicious'),
    summarize_years(X_train, train_years_not_found_ids, 'Train', 'Not found'),
    summarize_years(X_test, test_years_not_found_ids, 'Test', 'Not found'),
])

display(year_summary)

,split,status,rows,rows_with_year,minimum_year,years_below_1900,maximum_year
0,Train,Resolved,265,265,1809.0,260,2022.0
1,Test,Resolved,85,85,1802.0,83,2020.0
2,Train,Suspicious,160,0,NaN,0,NaN
3,Test,Suspicious,41,0,NaN,0,NaN
4,Train,Not found,228,0,NaN,0,NaN
5,Test,Not found,56,0,NaN,0,NaN


The summary below reports the lookup results for the current data and separates resolved, suspicious, and not-found IDs. Counts can change if the source data or API responses change. Missing years are not automatically imputed here. If you later impute them using artist-level statistics, calculate those statistics from training data only to avoid leaking test-set information.

<h3 style="color: #597edb; font-weight: 400;">2.2 Lyrics normalization <a id="lyrics-normalization"></a></h3>

Normalize the `lyrics` text in each frame before modeling while preserving accents, non-Latin scripts, emoji, and meaningful punctuation. The project helper `normalize_lyrics_inplace` in `preprocessing.py` applies `normalize_lyrics` from `text_quality.py` to the `lyrics` column and updates the supplied DataFrame in place. Use it on both splits:

```python
from preprocessing import normalize_lyrics_inplace

normalize_lyrics_inplace(X_train)
normalize_lyrics_inplace(X_test)
```

This is conservative text normalization, not task-specific tokenization, stopword removal, or stemming. Keep the original source files unchanged and compare any more aggressive transformations separately for each modeling objective.

In [ ]:
from preprocessing import normalize_lyrics_inplace

normalize_lyrics_inplace(X_train)
normalize_lyrics_inplace(X_test)

In [13]:
X_train.to_csv('data/normalized_x_train.csv', index=False)
X_test.to_csv('data/normalized_x_test.csv', index=False)

<h3 style="color: #597edb; font-weight: 400;">2.3 Duplicate-lyrics audit <a id="duplicate-audit"></a></h3>

Run this audit after normalizing both lyrics columns. It reports repeated non-empty lyrics within each dataset and exact normalized lyric matches between train and test. Normalization may change duplicate counts by making previously different formatting equivalent. The function reports matches without deleting or reordering rows.

In [ ]:
import importlib
import utils
importlib.reload(utils)
from utils import check_for_duplicates

duplicate_report = check_for_duplicates(
    X_train,
    X_test,
    dataset1_name="normalized_train",
    dataset2_name="normalized_test",
)
display(duplicate_report["summary"])

# Shared exact rows are available for inspection in duplicate_report["shared_rows"].